# LightFM Hybrid Baseline (Warm-Start)

Debug-friendly notebook version of `lightfm_hybrid_baseline.py`.

This pipeline supports parquet/csv, builds item metadata features, trains LightFM with WARP, and reports custom HR@10 + NDCG@10.


In [5]:
# Optional installs if missing:
# %pip install lightfm pyarrow
%pip install lightfm-next pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
"""
LightFM hybrid baseline for warm-start Amazon Fashion recommendation.

Key properties:
- Supports CSV and Parquet inputs (with automatic fallback and clear errors).
- Uses LightFM Dataset mapping for string IDs.
- Builds sparse item feature matrix from selected metadata signals:
  - leaf_category (high precision taxonomy signal)
  - main_categories (coarse but useful semantic grouping)
- Trains WARP model for ranking.
- Computes custom HR@10 and NDCG@10 for apples-to-apples comparison with ALS baselines.
"""

from __future__ import annotations

import argparse
import ast
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

try:
    from lightfm import LightFM
    from lightfm.data import Dataset
except Exception as exc:  # pragma: no cover - environment-specific dependency path
    raise ImportError(
        "lightfm is required. Install with: pip install lightfm"
    ) from exc


@dataclass
class HybridConfig:
    train_path: str = "mini_train.parquet"
    test_path: str = "mini_test.parquet"
    meta_path: str = "mini_meta.parquet"
    file_format: str = "auto"  # auto | parquet | csv

    user_col: str = "user_id"
    item_col: str = "parent_asin"
    rating_col: str = "rating"

    # Interaction policy for WARP: use only strong/positive events.
    min_positive_rating: float = 4.0
    test_positive_rating: float = 4.0

    # Metadata features: prioritize quality over quantity.
    use_leaf_category: bool = True
    use_main_categories: bool = True
    max_main_category_tokens: int = 3

    # LightFM params
    no_components: int = 64
    learning_rate: float = 0.05
    item_alpha: float = 1e-6
    user_alpha: float = 1e-6
    epochs: int = 30
    num_threads: int = 8
    random_state: int = 42

    # Eval
    k: int = 10


class LightFMHybridBaseline:
    """Hybrid LightFM baseline with custom HR@K/NDCG@K evaluation."""

    def __init__(self, cfg: HybridConfig) -> None:
        self.cfg = cfg
        self.dataset: Dataset | None = None
        self.model: LightFM | None = None

        self.user_id_map: Dict[str, int] = {}
        self.item_id_map: Dict[str, int] = {}

        self.train_interactions: csr_matrix | None = None
        self.item_features: csr_matrix | None = None

        self.train_df: pd.DataFrame | None = None
        self.test_df: pd.DataFrame | None = None
        self.meta_df: pd.DataFrame | None = None

    # ---------- IO ----------
    def _resolve_existing_path(self, path: str) -> Path:
        raw = Path(path)

        candidates: List[Path] = []
        if raw.is_absolute():
            candidates.append(raw)
        else:
            cwd = Path.cwd()
            candidates.extend([
                cwd / raw,
                cwd / raw.name,
                cwd / "research" / raw,
                cwd / "research" / raw.name,
            ])
            for parent in cwd.parents:
                candidates.extend([
                    parent / raw,
                    parent / raw.name,
                    parent / "research" / raw,
                    parent / "research" / raw.name,
                ])

        expanded: List[Path] = []
        for cand in candidates:
            expanded.append(cand)
            if cand.suffix.lower() == ".parquet":
                expanded.append(cand.with_suffix(".csv"))
            elif cand.suffix.lower() == ".csv":
                expanded.append(cand.with_suffix(".parquet"))

        seen = set()
        deduped: List[Path] = []
        for cand in expanded:
            key = str(cand)
            if key in seen:
                continue
            seen.add(key)
            deduped.append(cand)

        for cand in deduped:
            if cand.exists():
                return cand

        checked = "\n".join(f"- {c}" for c in deduped[:12])
        raise FileNotFoundError(
            f"File not found: {path}.\n"
            f"Current working directory: {Path.cwd()}\n"
            f"Checked:\n{checked}"
        )

    def _detect_format(self, path: Path) -> str:
        if self.cfg.file_format in {"csv", "parquet"}:
            return self.cfg.file_format
        suffix = path.suffix.lower()
        if suffix == ".parquet":
            return "parquet"
        if suffix == ".csv":
            return "csv"
        raise ValueError(
            f"Unable to infer file format for {path}. "
            "Set --file-format to csv or parquet."
        )

    def _read_table(self, path_str: str) -> pd.DataFrame:
        path = self._resolve_existing_path(path_str)
        fmt = self._detect_format(path)

        if fmt == "csv":
            return pd.read_csv(path)

        # parquet path
        try:
            return pd.read_parquet(path)
        except ImportError as exc:
            raise ImportError(
                "Parquet requested but pandas parquet engine is unavailable. "
                "Install pyarrow or fastparquet."
            ) from exc

    # ---------- Preprocessing ----------
    def _prepare_interactions(self, df: pd.DataFrame) -> pd.DataFrame:
        cfg = self.cfg
        required = [cfg.user_col, cfg.item_col, cfg.rating_col]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"Missing interaction columns: {missing}")

        out = df[required].copy()
        out[cfg.user_col] = out[cfg.user_col].astype(str)
        out[cfg.item_col] = out[cfg.item_col].astype(str)
        out[cfg.rating_col] = pd.to_numeric(out[cfg.rating_col], errors="coerce")
        out = out.dropna(subset=required)

        # Collapse duplicate user-item rows to a single strongest signal.
        out = out.groupby([cfg.user_col, cfg.item_col], as_index=False)[cfg.rating_col].max()
        return out

    def _prepare_metadata(self, df: pd.DataFrame) -> pd.DataFrame:
        cfg = self.cfg
        required = [cfg.item_col, "main_categories", "leaf_category"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"Missing metadata columns: {missing}")

        out = df[[cfg.item_col, "main_categories", "leaf_category"]].copy()
        out[cfg.item_col] = out[cfg.item_col].astype(str)
        out = out.drop_duplicates(subset=[cfg.item_col], keep="first")
        return out

    def load(self) -> None:
        self.train_df = self._prepare_interactions(self._read_table(self.cfg.train_path))
        self.test_df = self._prepare_interactions(self._read_table(self.cfg.test_path))
        self.meta_df = self._prepare_metadata(self._read_table(self.cfg.meta_path))

    # ---------- Feature engineering ----------
    @staticmethod
    def _normalize_token(token: str) -> str:
        token = token.strip().lower()
        token = re.sub(r"\s+", "_", token)
        token = re.sub(r"[^a-z0-9_]+", "", token)
        return token

    def _parse_main_categories(self, value: object) -> List[str]:
        if pd.isna(value):
            return []

        parsed_vals: Iterable[object]
        if isinstance(value, (list, tuple, set)):
            parsed_vals = value
        else:
            text = str(value).strip()
            if not text:
                return []

            # Try list-like string first; fallback to delimiter split.
            try:
                lit = ast.literal_eval(text)
                if isinstance(lit, (list, tuple, set)):
                    parsed_vals = lit
                else:
                    parsed_vals = [lit]
            except Exception:
                parsed_vals = re.split(r"[|,;/>\-]", text)

        cleaned: List[str] = []
        for raw in parsed_vals:
            tok = self._normalize_token(str(raw))
            if tok:
                cleaned.append(tok)

        # Keep top-N tokens to avoid noisy, overly dense feature spaces.
        return cleaned[: self.cfg.max_main_category_tokens]

    def _item_feature_tokens(self, row: pd.Series) -> List[str]:
        features: List[str] = []
        if self.cfg.use_leaf_category:
            leaf = self._normalize_token(str(row.get("leaf_category", "")))
            if leaf:
                features.append(f"leaf:{leaf}")

        if self.cfg.use_main_categories:
            for mc in self._parse_main_categories(row.get("main_categories", None)):
                features.append(f"main:{mc}")

        if not features:
            features.append("__no_meta__")
        return features

    # ---------- Dataset + Matrices ----------
    def _build_dataset_and_matrices(self) -> Tuple[csr_matrix, csr_matrix]:
        assert self.train_df is not None and self.test_df is not None and self.meta_df is not None
        cfg = self.cfg

        # Warm-start context: fit mappings on train entities.
        train_users = self.train_df[cfg.user_col].unique().tolist()
        train_items = self.train_df[cfg.item_col].unique().tolist()

        meta_indexed = self.meta_df.set_index(cfg.item_col, drop=False)
        item_feature_rows: List[Tuple[str, List[str]]] = []
        feature_vocab = {"__no_meta__"}

        for item_id in train_items:
            if item_id in meta_indexed.index:
                row = meta_indexed.loc[item_id]
                # If duplicate index returns DataFrame, take first.
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                feats = self._item_feature_tokens(row)
            else:
                feats = ["__no_meta__"]
            item_feature_rows.append((item_id, feats))
            feature_vocab.update(feats)

        ds = Dataset()
        ds.fit(
            users=train_users,
            items=train_items,
            item_features=list(feature_vocab),
        )
        self.dataset = ds

        # Build positive-only interactions for WARP ranking.
        train_pos = self.train_df[self.train_df[cfg.rating_col] >= cfg.min_positive_rating]
        if train_pos.empty:
            raise ValueError(
                f"No train interactions with rating >= {cfg.min_positive_rating}. "
                "Lower min_positive_rating or inspect data."
            )

        interaction_tuples = list(
            zip(
                train_pos[cfg.user_col].astype(str),
                train_pos[cfg.item_col].astype(str),
                np.ones(len(train_pos), dtype=np.float32),
            )
        )

        interactions, _weights = ds.build_interactions(interaction_tuples)
        interactions = interactions.tocsr()

        item_features = ds.build_item_features(item_feature_rows, normalize=True).tocsr()

        self.train_interactions = interactions
        self.item_features = item_features

        user_id_map, _user_feat_map, item_id_map, _item_feat_map = ds.mapping()
        self.user_id_map = user_id_map
        self.item_id_map = item_id_map

        return interactions, item_features

    # ---------- Training ----------
    def train(self) -> None:
        interactions, item_features = self._build_dataset_and_matrices()
        cfg = self.cfg

        model = LightFM(
            loss="warp",
            no_components=cfg.no_components,
            learning_rate=cfg.learning_rate,
            item_alpha=cfg.item_alpha,
            user_alpha=cfg.user_alpha,
            random_state=cfg.random_state,
        )
        model.fit(
            interactions=interactions,
            item_features=item_features,
            epochs=cfg.epochs,
            num_threads=cfg.num_threads,
            verbose=False,
        )
        self.model = model

    # ---------- Evaluation ----------
    def _build_test_positive_matrix(self) -> csr_matrix:
        assert self.train_interactions is not None and self.test_df is not None
        cfg = self.cfg

        test_pos = self.test_df[self.test_df[cfg.rating_col] >= cfg.test_positive_rating]
        if test_pos.empty:
            raise ValueError(
                f"No test interactions with rating >= {cfg.test_positive_rating}. "
                "Lower test_positive_rating or inspect data."
            )

        rows: List[int] = []
        cols: List[int] = []

        # Warm-start guaranteed by routing, but we still guard and skip unknowns.
        for r in test_pos.itertuples(index=False):
            uid = str(getattr(r, cfg.user_col))
            iid = str(getattr(r, cfg.item_col))
            uidx = self.user_id_map.get(uid)
            iidx = self.item_id_map.get(iid)
            if uidx is None or iidx is None:
                continue
            rows.append(uidx)
            cols.append(iidx)

        if not rows:
            raise ValueError("No mappable warm-start test positives after ID mapping.")

        data = np.ones(len(rows), dtype=np.int32)
        shape = self.train_interactions.shape
        return coo_matrix((data, (rows, cols)), shape=shape, dtype=np.int32).tocsr()

    def evaluate_hr_ndcg(self) -> Dict[str, float]:
        assert self.model is not None
        assert self.train_interactions is not None and self.item_features is not None

        k = self.cfg.k
        test_pos_matrix = self._build_test_positive_matrix()

        # Efficiently compute item ranks for only test-positive entries.
        # This avoids full user-item score materialization.
        rank_matrix = self.model.predict_rank(
            test_interactions=test_pos_matrix,
            train_interactions=self.train_interactions,
            item_features=self.item_features,
            num_threads=self.cfg.num_threads,
            check_intersections=False,
        ).tocsr()

        hr_scores: List[float] = []
        ndcg_scores: List[float] = []

        for u in range(rank_matrix.shape[0]):
            start, end = rank_matrix.indptr[u], rank_matrix.indptr[u + 1]
            if start == end:
                continue

            ranks = rank_matrix.data[start:end]  # 0-based ranks
            # HR@K: any relevant item appears in top-K.
            hit = np.any(ranks < k)
            hr_scores.append(float(hit))

            # NDCG@K with binary relevance on warm test positives.
            topk_ranks = ranks[ranks < k]
            if topk_ranks.size == 0:
                ndcg_scores.append(0.0)
                continue

            positions = topk_ranks + 1  # convert to 1-based positions
            dcg = float(np.sum(1.0 / np.log2(positions + 1)))

            rel_count = ranks.size
            ideal_len = min(rel_count, k)
            idcg = float(np.sum(1.0 / np.log2(np.arange(2, ideal_len + 2))))
            ndcg_scores.append(dcg / idcg if idcg > 0 else 0.0)

        if not hr_scores:
            return {"HR@10": 0.0, "NDCG@10": 0.0, "eval_users": 0}

        return {
            "HR@10": float(np.mean(hr_scores)),
            "NDCG@10": float(np.mean(ndcg_scores)),
            "eval_users": int(len(hr_scores)),
        }

    def run(self) -> Dict[str, float]:
        self.load()
        self.train()
        return self.evaluate_hr_ndcg()


In [7]:
# Update paths and params here for debugging / experiments
cfg = HybridConfig(
    train_path="mini_train.csv",
    test_path="mini_test.csv",
    meta_path="mini_meta.csv",
    file_format="auto",
    min_positive_rating=4.0,
    test_positive_rating=4.0,
    no_components=64,
    epochs=30,
    num_threads=8,
)

pipeline = LightFMHybridBaseline(cfg)
metrics = pipeline.run()
metrics


{'HR@10': 0.015880017644464048,
 'NDCG@10': 0.00849536309239001,
 'eval_users': 11335}

In [8]:
# Optional debug: inspect mapped dimensions and sparsity
print("train interactions shape:", pipeline.train_interactions.shape)
print("train nnz:", pipeline.train_interactions.nnz)
print("item features shape:", pipeline.item_features.shape)
print("item feature nnz:", pipeline.item_features.nnz)
print("mapped users:", len(pipeline.user_id_map))
print("mapped items:", len(pipeline.item_id_map))


train interactions shape: (219824, 135409)
train nnz: 199965
item features shape: (135409, 135851)
item feature nnz: 406227
mapped users: 219824
mapped items: 135409
